<a href="https://colab.research.google.com/github/Ashutosh1910/indic-watermarking/blob/main/prompt-set/promptset/notebooks/translate_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndicTrans2 1B translation of the prompt set (Colab / Kaggle GPU)

Same model, beams and script as the local run (`promptset/scripts/translate_indictrans2.py`, 1B, 5 beams), on a free T4 GPU.
Runtime > Change runtime type > T4 GPU (Kaggle: Accelerator GPU T4, Internet on).

Needs a Hugging Face token from an account that has accepted the IndicTrans2 terms
(`ai4bharat/indictrans2-en-indic-1B` and `ai4bharat/indictrans2-indic-en-1B`).

To resume, upload the `translated.jsonl` you already have in step 3: rows already in it are skipped.

In [ ]:
!pip -q install "transformers==4.57.6" "IndicTransToolkit==1.1.1" sentencepiece sacremoses
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from huggingface_hub import login
login()  # paste a read token

In [ ]:
!wget -q -O to_translate.jsonl https://raw.githubusercontent.com/Ashutosh1910/indic-watermarking/claude/prompt-set/promptset/data/work/to_translate.jsonl
!wget -q -O translate_indictrans2.py https://raw.githubusercontent.com/Ashutosh1910/indic-watermarking/claude/prompt-set/promptset/scripts/translate_indictrans2.py
!wc -l to_translate.jsonl
# Optional: resume from a partial translated.jsonl (Colab only; on Kaggle add it as a dataset and copy it here)
try:
    from google.colab import files
    up = files.upload()  # pick translated.jsonl, or cancel to start fresh
except Exception as e:
    print("no upload:", e)

In [ ]:
!python translate_indictrans2.py --in to_translate.jsonl --out translated.jsonl --size 1B --batch 32 --beams 5 --device cuda

In [ ]:
import json
src = [json.loads(l) for l in open("to_translate.jsonl") if l.strip()]
out = [json.loads(l) for l in open("translated.jsonl") if l.strip()]
ids = {r["id"] for r in out}
print(len(src), "inputs,", len(out), "rows,", len(ids), "unique ids")
print("missing:", [r["id"] for r in src if r["id"] not in ids][:10])
print("empty:", [r["id"] for r in out if not r["translation"].strip()][:10])
for r in out[:3]:
    print(r["src"], "->", r["tgt"], "|", r["text"][:120], "=>", r["translation"][:160])

In [ ]:
try:
    from google.colab import files
    files.download("translated.jsonl")
except Exception:
    print("Kaggle: download translated.jsonl from the Output panel")